In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer

# Load datasets
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
misconception_mapping = pd.read_csv('misconception_mapping.csv')
sample_submission = pd.read_csv('sample_submission.csv')

# Preprocess text fields
def preprocess_text(text):
    if isinstance(text, str):
        return text.lower()
    return ""

train_df['QuestionText'] = train_df['QuestionText'].apply(preprocess_text)
for col in ['AnswerAText', 'AnswerBText', 'AnswerCText', 'AnswerDText']:
    train_df[col] = train_df[col].apply(preprocess_text)

# Initialize MultiLabelBinarizer
mlb = MultiLabelBinarizer()

# Prepare labels
labels = train_df[['MisconceptionAId', 'MisconceptionBId', 'MisconceptionCId']].applymap(
    lambda x: [int(x)] if not pd.isna(x) else []
)
labels = labels.apply(lambda row: [item for sublist in row for item in sublist], axis=1)
labels = mlb.fit_transform(labels)

# Ensure CombinedText creation aligns with labels
train_df['CombinedText'] = train_df['QuestionText'] + " " + train_df['AnswerAText'] + " " + \
                          train_df['AnswerBText'] + " " + train_df['AnswerCText'] + " " + train_df['AnswerDText']

# Verify shape alignment
assert len(train_df['CombinedText']) == labels.shape[0], "Mismatch between text and labels."

# Split data for validation
X_train, X_val, y_train, y_val = train_test_split(train_df['CombinedText'], labels, test_size=0.2, random_state=42)

# Model pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('clf', MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42)))
])

# Train the model
pipeline.fit(X_train, y_train)

# Validate the model
y_val_pred_list = pipeline.predict_proba(X_val)

# Dynamically handle single-class outputs
y_val_pred = np.array([
    pred[:, 1] if pred.shape[1] > 1 else pred[:, 0]
    for pred in y_val_pred_list
]).T

y_val_pred_binary = (y_val_pred > 0.5).astype(int)

# Evaluate using MAP@25
def map_at_k(y_true, y_pred, k=25):
    map_total = 0.0
    valid_rows = 0  # Count rows where y_true has at least one relevant label
    
    for i in range(len(y_true)):
        y_true_row = y_true[i]
        y_pred_row = y_pred[i]
        
        # Skip rows with no relevant labels
        if np.sum(y_true_row) == 0:
            continue
        
        sorted_indices = np.argsort(y_pred_row)[::-1][:k]
        score = 0.0
        num_relevant = 0
        
        for idx, pred_idx in enumerate(sorted_indices):
            if y_true_row[pred_idx] == 1:
                num_relevant += 1
                score += num_relevant / (idx + 1)
        
        map_total += score / min(k, np.sum(y_true_row))
        valid_rows += 1
    
    return map_total / valid_rows if valid_rows > 0 else float('nan')

map_score = map_at_k(y_val, y_val_pred, k=25)
print(f"Validation MAP@25: {map_score}")

# Predict for test set
test_df['CombinedText'] = test_df['QuestionText'] + " " + test_df['AnswerAText'] + " " + \
                         test_df['AnswerBText'] + " " + test_df['AnswerCText'] + " " + test_df['AnswerDText']

X_test = test_df['CombinedText']
y_test_pred_list = pipeline.predict_proba(X_test)

# Dynamically handle single-class outputs for test predictions
y_test_pred = np.array([
    pred[:, 1] if pred.shape[1] > 1 else pred[:, 0]
    for pred in y_test_pred_list
]).T

# Function to prepare the submission file
def prepare_submission(test_df, y_pred, mlb):
    predictions = []
    
    # Itérer sur chaque ligne dans le DataFrame de test
    for idx, row in test_df.iterrows():
        question_id = row['QuestionId']
        
        # Pour chaque option de réponse A, B, C, D
        for i, answer in enumerate(['A', 'B', 'C', 'D']):
            # Créer la valeur 'QuestionId_Answer' (ex : 1869_A)
            question_answer = f"{question_id}_{answer}"
            
            # Obtenir les prédictions pour chaque réponse (A, B, C, D)
            # Ici on suppose que y_pred[idx] est un tableau avec 4 valeurs pour chaque question
            # correspondant aux réponses A, B, C, D respectivement
            y_pred_answer = y_pred[idx, i]  # Prédiction spécifique à la réponse A, B, C, ou D
            
            # Obtenir les indices des 25 meilleures prédictions pour cette réponse
            sorted_indices = np.argsort(y_pred_answer)[::-1][:25]
            misconceptions = " ".join(map(str, mlb.classes_[sorted_indices]))
            
            # Ajouter la ligne de prédiction
            predictions.append([question_answer, misconceptions])
    
    # Créer un DataFrame avec les prédictions
    submission_df = pd.DataFrame(predictions, columns=['QuestionId_Answer', 'MisconceptionId'])
    
    return submission_df


submission = prepare_submission(test_df, y_test_pred, mlb)

# Sauvegarder le fichier de soumission
submission.to_csv('submission.csv', index=False)

print("Le fichier de soumission a été sauvegardé sous le nom 'submission.csv'")



Validation MAP@25: 0.007091291257137019
Le fichier de soumission a été sauvegardé sous le nom 'submission.csv'
